# Retrieval Component

## 1. Introduction

### 1.1. Install and Import Libraries

In [ ]:
# %pip install -U pandas numpy tqdm python-dotenv neo4j qdrant-client rank-bm25
# %pip install -U FlagEmbedding sentence-transformers torch

In [10]:
# ============================================
# 1. Standard Library
# ============================================
import gc
import json
import math
import os
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from importlib import reload
from pathlib import Path
from typing import Any, Iterable, Mapping

# ============================================
# 2. Third-party Libraries
# ============================================
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client import models as qdrant_models
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

def l2_normalize_matrix(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """L2-normalize rows without depending on scikit-learn."""
    array = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(array, axis=1, keepdims=True)
    return array / np.maximum(norms, eps)

# ============================================
# 3. Project Helpers
# ============================================
# Keep startup imports light. BGE-M3 and LLM helpers import Transformers, which can
# trip over local binary package issues; load them lazily in the cells that need them.
import src.graph_kb_neo4j as neo4j_helpers
import src.retrieval_drivers as retrieval_driver_helpers
import src.retrieval_sparse as sparse_retrieval_helpers
# import src.retrieval_query_generation as query_generation_helpers

reload(neo4j_helpers)
reload(retrieval_driver_helpers)
reload(sparse_retrieval_helpers)
# reload(query_generation_helpers)

# from src.retrieval_query_generation import load_generated_qrels, read_jsonl
from src.retrieval_drivers import Neo4jGraphRetrievalDriver, QdrantRetrievalDriver
from src.retrieval_sparse import ArchiveChunkCorpus, BM25SparseRetriever

# Lazy imports to use later when needed:
# import src.bge_m3_embedding as bge_m3_helpers
# reload(bge_m3_helpers)
# from src.bge_m3_embedding import BGEM3Embedder
#
# import src.retrieval_llms as retrieval_llm_helpers
# reload(retrieval_llm_helpers)
# from src.retrieval_llms import RelevanceJudgeLLM, extract_first_json_object


### 1.2. Environment Setup & Service Initialization

#### 1.2.1. Google Colab Setup

Run this section if you are running this notebook on Google Colab. Otherwise, run section 1.3.2.

In [ ]:
# from google.colab import userdata

In [ ]:
# QDRANT_CLUSTER_URL = userdata.get("QDRANT_CLUSTER_URL")
# QDRANT_CLUSTER_API_KEY = userdata.get("QDRANT_CLUSTER_API_KEY")
# COLLECTION_NAME = "archive-chunks"
#
# qdrant_client = QdrantClient(
#     url=QDRANT_CLUSTER_URL,
#     api_key=QDRANT_CLUSTER_API_KEY,
# )

# print(qdrant_client.get_collections())

In [ ]:
# NEO4J_URI = userdata.get("NEO4J_URI")
# NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
# NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")

# driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
# driver.verify_connectivity()

#### 1.2.2. Local Setup

In [ ]:
load_dotenv()

QDRANT_CLUSTER_URL = os.getenv("QDRANT_CLUSTER_URL")
QDRANT_CLUSTER_API_KEY = os.getenv("QDRANT_CLUSTER_API_KEY")
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION_NAME", "archive-chunks")

qdrant_retrieval_driver = QdrantRetrievalDriver(
    url=QDRANT_CLUSTER_URL,
    api_key=QDRANT_CLUSTER_API_KEY,
    collection_name=COLLECTION_NAME,
)

print(qdrant_retrieval_driver.verify_connectivity())

In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None

neo4j_graph_driver = Neo4jGraphRetrievalDriver(
    uri=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
neo4j_graph_driver.verify_connectivity()
print("Neo4j graph retrieval driver connected.")

### 1.3. Initialize LLM Services

In [ ]:
INITIALIZE_RELEVANCE_JUDGE_LLM = True

if INITIALIZE_RELEVANCE_JUDGE_LLM:
    relevance_judge_llm = RelevanceJudgeLLM(
        token=HUGGING_FACE_TOKEN,
        torch_dtype=LLM_TORCH_DTYPE,
        load_in_4bit=LLM_LOAD_IN_4BIT,
    )
    print(f"Loaded relevance-judge LLM on: {relevance_judge_llm.device}")
else:
    relevance_judge_llm = None
    print("Relevance judge initialization skipped for now.")

## 2. Tune and Smoke Test Retrieval Strategies

In [11]:
RETRIEVAL_EXPORT_DIR = Path("data/evaluation/retrieval_runs")
RETRIEVAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_CHUNKS_CSV = Path("data/graph_kb_exports/step_01_archive_skeleton/chunks.csv")
RETRIEVAL_SMOKE_TEST_QUERY_ID = "smoke_query_001"
RETRIEVAL_SMOKE_TEST_QUERY = "What concern did legal scholars raise about CIA drone strikes?"
RETRIEVAL_SMOKE_TEST_TOP_K = 10

print(RETRIEVAL_SMOKE_TEST_QUERY)

What concern did legal scholars raise about CIA drone strikes?


### 2.1. Sparse Retrieval

#### 2.1.1. Tune top-k

In [16]:
from importlib import reload

import src.retrieval_sparse as sparse_retrieval_helpers
reload(sparse_retrieval_helpers)

from src.retrieval_sparse import ArchiveChunkCorpus, BM25SparseRetriever

In [17]:
# TODO: tune sparse retrieval top-k after generated qrels are ready.
# This cell will evaluate candidate top-k values using only query_id/query for retrieval
# and qrel fields only after retrieval for evaluation.
SPARSE_RETRIEVAL_TOP_K_CANDIDATES = [5, 10, 20, 50]
BEST_SPARSE_RETRIEVAL_TOP_K = RETRIEVAL_SMOKE_TEST_TOP_K
print(f"Using smoke-test sparse top-k: {BEST_SPARSE_RETRIEVAL_TOP_K}")

Using smoke-test sparse top-k: 10


#### 2.1.2. Smoke test

In [19]:
SPARSE_RETRIEVAL_MAX_CHUNKS = None
SPARSE_RETRIEVAL_CHUNKSIZE = 50_000
SPARSE_RETRIEVAL_FORCE_REBUILD = False
BM25_CACHE_PATH = Path("data/evaluation/retrieval_indexes/bm25_sparse_retriever.pkl")

if "bm25_sparse_retriever" not in globals() or SPARSE_RETRIEVAL_FORCE_REBUILD:
    bm25_sparse_retriever = BM25SparseRetriever.from_cache_or_build(
        ARCHIVE_CHUNKS_CSV,
        cache_path=BM25_CACHE_PATH,
        max_chunks=SPARSE_RETRIEVAL_MAX_CHUNKS,
        chunksize=SPARSE_RETRIEVAL_CHUNKSIZE,
        force_rebuild=SPARSE_RETRIEVAL_FORCE_REBUILD,
        verbose=True,
    )
    archive_chunk_corpus = bm25_sparse_retriever.corpus

print(f"Sparse retrieval corpus chunks: {len(bm25_sparse_retriever.corpus.chunks_df):,}")
print(f"BM25 cache path: {BM25_CACHE_PATH}")

sparse_smoke_results_df = bm25_sparse_retriever.retrieve(
    RETRIEVAL_SMOKE_TEST_QUERY,
    query_id=RETRIEVAL_SMOKE_TEST_QUERY_ID,
    top_k=BEST_SPARSE_RETRIEVAL_TOP_K,
)

display(sparse_smoke_results_df[[
    "rank",
    "score",
    "chunk_id",
    "dataset",
    "modality",
    "title",
    "retrieval_text_preview",
]])

Sparse retrieval corpus chunks: 297,016
BM25 cache path: data\evaluation\retrieval_indexes\bm25_sparse_retriever.pkl


,rank,score,chunk_id,dataset,modality,title,retrieval_text_preview
0,1,38.879647,cda827e281f3e728d26877de,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,CNN/DailyMail Article 0348c10a8212\nNEW: ACLU ...
1,2,37.134493,e6be75d4356e2c23cc1e1370,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,CNN/DailyMail Article 0348c10a8212\nNEW: ACLU ...
2,3,36.925838,c1b57b6fb5f8f31d40bb59a0,cnn_dailymail,article,CNN/DailyMail Article f7a2b975f3df,CNN/DailyMail Article f7a2b975f3df\nPresident ...
3,4,35.772750,67e9541f255761fe88837d03,cnn_dailymail,article,CNN/DailyMail Article 2fe159232cbb,CNN/DailyMail Article 2fe159232cbb\nJohn Brenn...
4,5,35.550204,c4af52224173b8d7d292caf6,cnn_dailymail,article,CNN/DailyMail Article 8bdd40c8e9d7,CNN/DailyMail Article 8bdd40c8e9d7\nPakistan c...
5,6,35.429063,32e749047c6217008d8350ab,cnn_dailymail,article,CNN/DailyMail Article 2cdbffd07118,CNN/DailyMail Article 2cdbffd07118\nNEW: A U.S...
6,7,34.748995,4affde0cd0eec63f8b897038,cnn_dailymail,article,CNN/DailyMail Article 0adbcedec99d,"CNN/DailyMail Article 0adbcedec99d\nNEW: ""Do t..."
7,8,34.571345,86639d086be1756335276c79,cnn_dailymail,article,CNN/DailyMail Article a7fd3e0994ac,CNN/DailyMail Article a7fd3e0994ac\nWhat is th...
8,9,34.476715,ce54fee5f5a829b8709c233b,cnn_dailymail,article,CNN/DailyMail Article 0adbcedec99d,"CNN/DailyMail Article 0adbcedec99d\nNEW: ""Do t..."
9,10,34.349359,d6a39f368e575710d006f305,cnn_dailymail,article,CNN/DailyMail Article a7fd3e0994ac,CNN/DailyMail Article a7fd3e0994ac\nWhat is th...


In [ ]:
# Inspect the full text for the top sparse retrieval results.
SPARSE_SMOKE_PREVIEW_TOP_N = 3

if "sparse_smoke_results_df" not in globals() or sparse_smoke_results_df.empty:
    raise RuntimeError("Run the sparse smoke-test retrieval cell first.")

top_sparse_chunk_ids = sparse_smoke_results_df.head(SPARSE_SMOKE_PREVIEW_TOP_N)["chunk_id"].tolist()
top_sparse_texts_df = (
    bm25_sparse_retriever.corpus.chunks_df[
        bm25_sparse_retriever.corpus.chunks_df["chunk_id"].isin(top_sparse_chunk_ids)
    ][["chunk_id", "document_id", "dataset", "modality", "title", "retrieval_text"]]
    .merge(
        sparse_smoke_results_df[["chunk_id", "rank", "score"]],
        on="chunk_id",
        how="inner",
    )
    .sort_values("rank")
)

for _, row in top_sparse_texts_df.iterrows():
    print("=" * 100)
    print(f"Rank {int(row['rank'])} | score={row['score']:.4f} | chunk_id={row['chunk_id']}")
    print(f"Dataset={row['dataset']} | Modality={row['modality']} | Document={row['document_id']}")
    print(f"Title: {row['title']}")
    print("-" * 100)
    print(str(row["retrieval_text"])[:3000])
    print()